In [ ]:
# =========================================================
# CELL 1: CẤU HÌNH LIÊN KẾT MÔ-ĐUN VÀ IMPORT THƯ VIỆN
# =========================================================
import os
import sys
import gradio as gr

# Ép hệ thống nhận diện thư mục gốc để tìm thấy mô-đun src.predict
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

try:
    from src.predict import predict_sentiment
    print("✅ Liên kết Backend predict.py thành công!")
except ImportError:
    # Phòng trường hợp chạy trực tiếp từ thư mục chứa file demo
    sys.path.append('.')
    from src.predict import predict_sentiment
    print("✅ Liên kết Backend predict.py thành công (đường dẫn dự phòng)!")

c:\NN PYTHON\project\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Đang nạp bộ trọng số PhoBERT Base hoàn toàn từ local...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1949.03it/s]

✅ Nạp mô hình PhoBERT thành công! Hệ thống đã sẵn sàng suy luận.
✅ Liên kết Backend predict.py thành công!


In [ ]:
# ======================================================
# CELL 2: ĐỊNH NGHĨA HÀM XỬ LÝ LOGIC CHO GIAO DIỆN
# ======================================================
def gradio_predict(text):
    """
    Hàm cầu nối nhận text từ Frontend Gradio, kích hoạt suy luận PhoBERT 
    ở Backend và trả về kết quả định dạng trực quan.
    """
    if not text or not text.strip():
        return "⚠️ Vui lòng nhập dữ liệu văn bản!", "", ""
    
    # Gọi hàm xử lý suy luận khép kín từ predict.py
    res = predict_sentiment(text)
    
    if res["label_id"] == -1:
        return "⚠️ Không có dữ liệu hợp lệ", "", ""
        
    # 1. Định dạng kết quả nhãn tối hậu
    out_label = f"✨ {res['label'].upper()} (Độ tin cậy: {res['confidence']:.2%})"
    
    # 2. Định dạng bảng phân phối xác suất trên dải 3 nhãn
    probs = res["probabilities"]
    out_probs = (
        f"🔴 Tiêu cực (Nhãn 0)  : {probs['Tiêu cực (Nhãn 0)']:.2%}\n"
        f"🟡 Trung tính (Nhãn 1) : {probs['Trung tính (Nhãn 1)']:.2%}\n"
        f"🟢 Tích cực (Nhãn 2)  : {probs['Tích cực (Nhãn 2)']:.2%}"
    )
    
    # 3. Định dạng thông số hạ tầng và kích thước Vector Tensor
    out_token = f"⚡ Kích thước Tensor: [{res['n_tokens']} / 256 Tokens] | Max Length đồng bộ hệ thống"
    
    return out_label, out_probs, out_token

In [ ]:
# ====================================================\n",
# CELL 3: CẤU HÌNH DANH MỤC CÂU VĂN THỰC NGHIỆM MẪU \n",
# ====================================================\n",
EXAMPLES = [
    ["giảng viên nhiệt tình , giáo trình rất hay và chi tiết ."],
    ["cách truyền đạt khó hiểu và bài tập quá nặng ."],
    ["thầy dạy lý thuyết tạm ổn ."],
    ["tạm thời chưa có ."],
    ["nếu yêu cầu bài tập thì có thể nộp giấy riêng cho cô ."], 
    ["cơ sở vật chất phòng học rất tốt, máy lạnh mát mẻ."],
    ["giảng viên thường xuyên đi trễ và ít tương tác với sinh viên."],
    ["dạy tương đối đầy đủ chương trình ."]
]
print(f"📊 Đã nạp thành công {len(EXAMPLES)} câu thực nghiệm mẫu phục vụ nghiệm thu.")

📊 Đã nạp thành công 8 câu thực nghiệm mẫu phục vụ nghiệm thu.


In [ ]:
# ==========================================================\n",
# CELL 4: THIẾT KẾ LAYOUT VÀ KÍCH HOẠT GIAO DIỆN GRADIO UI\n",
# ==========================================================\n",
# Sửa đổi: Loại bỏ tham số theme ra khỏi gr.Blocks để tương thích Gradio 6.0
with gr.Blocks(title="PhoBERT Sentiment Analytics") as demo:
    
    gr.Markdown(
        """
        # 🏫 PHÂN TÍCH SẮC THÁI PHẢN HỒI CỦA SINH VIÊN (HỆ THỐNG PHOBERT BASE)
        ### Đồ án Nghiên cứu & Ứng dụng Xử lý Ngôn ngữ Tự nhiên — Nhóm 4
        *Hệ thống tự động phân tách văn bản tiếng Việt thành 3 trạng thái sắc thái học thuật: Tích cực, Trung tính, Tiêu cực.*
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.Textbox(
                label="📝 Nhập câu văn phản hồi thô của sinh viên (Hệ thống tự động chuẩn hóa):", 
                placeholder="Ví dụ: Thầy cô giảng bài rất hay và tâm huyết...",
                lines=5
            )
            with gr.Row():
                btn_clear = gr.Button("🔄 Xóa sạch dòng", variant="secondary")
                btn_run = gr.Button("🎯 Phân tích cảm xúc", variant="primary")
                
        with gr.Column(scale=1):
            out_label = gr.Textbox(label="🏷️ Nhãn sắc thái cảm xúc tối hậu (Mô hình PhoBERT):", interactive=False)
            out_probs = gr.Textbox(label="📊 Phân phối xác suất trọn vẹn trên dải 3 nhãn:", interactive=False, lines=4)
            out_token = gr.Textbox(label="⚡ Thông số hạ tầng suy luận thời gian thực:", interactive=False)

    gr.Examples(examples=EXAMPLES, inputs=inp, label="Danh mục câu văn thực nghiệm mẫu phục vụ nghiệm thu:")

    btn_run.click(fn=gradio_predict, inputs=inp, outputs=[out_label, out_probs, out_token])
    inp.submit(fn=gradio_predict, inputs=inp, outputs=[out_label, out_probs, out_token])
    btn_clear.click(fn=lambda: ("", "", ""), outputs=[inp, out_label, out_probs, out_token])

if __name__ == "__main__":
    demo.launch(share=False, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
